# Periodic Analysis

If we are comparing all windows against a baseline, the result is largely dominated by the
difference in motion. Therefore, I am testing comparing period to period.

**This notebook currently covers steps 1-4 of the plan** (`period-comparison-plan.md`):
load, segment, check the segmentation visually, save it. It deliberately stops before any
distance computation, so the segmentation can be validated before spending compute on it.

In [21]:
# --- Imports and project paths -------------------------------------------------
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import yaml
from plotly.subplots import make_subplots

# Find the project root whether the kernel started in the repo root or in notebook/,
# then make both it and src/ importable. Must happen BEFORE the toolbox import below.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
for path in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from event_data_toolbox.event_data_manager import EventDataManager

# Plot colours, shared by every figure below.
INK, MUTED, BLUE, ORANGE = "#0b0b0b", "#52514e", "#2a78d6", "#eb6834"

print(f"project root: {PROJECT_ROOT}")

project root: c:\Users\cxm3593\Academic\Workspace\EventSimilarityAnalysis


## Step 1 - Pick a trial and read its calibration

Each trial folder has a `result.yaml` written by the EventCamCalib pipeline. Two numbers
from it matter here:

- `symmetry_period_us` - aperture-to-aperture, i.e. one optical modulation cycle.
- `rotation_period_us` - one full wheel revolution. With `apertures: 2` this is twice the
  symmetry period.

Segmenting on the **symmetry** period gives twice as many segments, but consecutive
segments then alternate between two physically distinct apertures. If those apertures
differ at all, that difference contaminates the real-vs-real noise floor. Switch
`PERIOD_KIND` to `"rotation"` to test that later.

In [22]:
# --- Trial selection -----------------------------------------------------------
TRIALS_DIR  = Path(r"C:/Users/cxm3593/Academic/Workspace/EventCamCalib/output/trials")
TRIAL       = "f1"          # f1 .. f5
PERIOD_KIND = "symmetry"    # "symmetry" or "rotation"

trial_dir = TRIALS_DIR / f"optical_chopper_data_{TRIAL}"
REAL_PATH = trial_dir / "final_masked_real.h5"
V2E_PATH  = trial_dir / "final_masked_v2e.h5"

with open(trial_dir / "result.yaml", "r") as f:
    CALIB = yaml.safe_load(f)["temporal_fine_calibration"]

PERIOD_US = int(round(CALIB[f"{PERIOD_KIND}_period_us"]))

print(f"trial            : {TRIAL}")
print(f"apertures        : {CALIB['apertures']}")
print(f"{PERIOD_KIND + ' period':17}: {PERIOD_US:,} us = {PERIOD_US/1000:.2f} ms "
      f"({1e6/PERIOD_US:.4f} Hz)")
print(f"real<->v2e offset: {CALIB['offset_us']:.1f} us  (already applied to the h5 files)")

trial            : f1
apertures        : 2
symmetry period  : 491,342 us = 491.34 ms (2.0352 Hz)
real<->v2e offset: -6159.0 us  (already applied to the h5 files)


## Step 2 - Recording time span

Fixed-period splitting needs only where the recording starts and ends, so nothing is read
into memory here. `load_event_data_h5` returns a lazy h5py dataset; because the events are
stored in time order, the first and last timestamps are two single-element reads.

On f1 that is 0.9 ms, against 3.7 s to pull all 22 M timestamps - about 4000x.

In [23]:
# --- Time span (two scalar reads, no bulk load) --------------------------------
data_manager = EventDataManager()
real_events = data_manager.load_event_data_h5(REAL_PATH, dataset_name="events", data_key="real_data")
v2e_events  = data_manager.load_event_data_h5(V2E_PATH,  dataset_name="events", data_key="v2e_data")

REAL_T_START, REAL_T_END = int(real_events[0]["t"]), int(real_events[-1]["t"])
V2E_T_START,  V2E_T_END  = int(v2e_events[0]["t"]),  int(v2e_events[-1]["t"])

print(f"real: {real_events.shape[0]:>12,} events   "
      f"t = {REAL_T_START:>8,} .. {REAL_T_END:>10,} us   ({(REAL_T_END-REAL_T_START)/1e6:.2f} s)")
print(f"v2e : {v2e_events.shape[0]:>12,} events   "
      f"t = {V2E_T_START:>8,} .. {V2E_T_END:>10,} us   ({(V2E_T_END-V2E_T_START)/1e6:.2f} s)")

real:   21,981,462 events   t =    6,159 .. 15,038,204 us   (15.03 s)
v2e :   58,685,494 events   t =        0 .. 15,032,434 us   (15.03 s)


## Step 3 - Split timestamps

A splitter returns the **cut points**: an array of `N + 1` timestamps defining `N` segments,
where segment `k` spans `splits[k] .. splits[k+1]`.

`events` is passed but unused by the fixed-period splitter. It is in the signature so a
later splitter that has to look at the data - flank crossings, blade angle - can read what
it needs lazily without changing how it is called.

Segmentation is derived from **real** only. v2e inherits the same cut points, so any v2e
timing error shows up as measured difference instead of being absorbed into the split.

In [24]:
# --- Splitters -----------------------------------------------------------------

def split_fixed_period(events, t_start_us, t_end_us, *, period_us):
    """Cut points every ``period_us`` across ``[t_start_us, t_end_us]``.

    Assumes the chopper period never varies. Reads no event data.
    Returns an ``int64`` array of N+1 timestamps; any leftover shorter than one
    full period at the end is dropped.
    """
    return np.arange(int(t_start_us), int(t_end_us) + 1, int(period_us), dtype=np.int64)


SPLITTERS = {
    "fixed_period": split_fixed_period,
    # "flank_crossing": split_flank_crossing,   # <- next method
}

In [25]:
# --- Generate the split timestamps ---------------------------------------------
SPLITTER_NAME   = "fixed_period"
SPLITTER_PARAMS = dict(period_us=PERIOD_US)

splits = SPLITTERS[SPLITTER_NAME](real_events, REAL_T_START, REAL_T_END, **SPLITTER_PARAMS)

print(f"Will split with {splits.shape[0]} cut points")
print(f"splitter   : {SPLITTER_NAME} {SPLITTER_PARAMS}")
print(f"cut points : {len(splits)}  ->  {len(splits) - 1} segments")
print(f"span       : {splits[0]:,} .. {splits[-1]:,} us  ({(splits[-1]-splits[0])/1e6:.2f} s)")
print(f"dropped    : {REAL_T_END - splits[-1]:,} us of tail "
      f"({(REAL_T_END - splits[-1]) / PERIOD_US:.2f} of a period)")
print(f"\nfirst 4: {splits[:4]}")

Will split with 31 cut points
splitter   : fixed_period {'period_us': 491342}
cut points : 31  ->  30 segments
span       : 6,159 .. 14,746,419 us  (14.74 s)
dropped    : 291,785 us of tail (0.59 of a period)

first 4: [   6159  497501  988843 1480185]


## Step 4 - Compare segments by the direction the events lie in

Counting events cannot check a split: with the wheel turning steadily the count stays
roughly the same wherever the cuts land. What can check it is *where* the events are.

For each event, take the direction it lies in as seen from the centre of the fitted ellipse.
Two segments cut at the same point in the cycle should give the same pattern of directions;
the difference between them is the split error, in degrees of wheel angle.

**Two corrections matter, and without them the measurement is wrong.**

*Directions cannot be averaged by adding them.* 350 and 10 are only 20 apart, but their
numeric average is 180 - the opposite side. Instead each direction is drawn as an arrow of
length one, the arrows are averaged, and the direction of the result is read off. Averaging
arrows means averaging their sideways and upward parts separately.

*A blade edge is a line, not an arrow.* Events fire along the whole edge, on both sides of
the centre, so an event at 30 and one at 210 describe the same blade. Doubling every
direction first makes those two land on the same value, because doubling 210 gives 420,
which is a full turn past 360 and therefore the same as 60. Without this the two ends of
the blade fight each other and produce a jump once per revolution that has nothing to do
with the wheel.

In [26]:
# --- Average arrow per window --------------------------------------------------
from scipy.stats import binned_statistic   # noqa: F401  (kept for the calibration code path)

WINDOW_US = 2500        # window length inside a segment
POLARITY  = "ON"        # "ON" or "OFF"
DOUBLE    = True        # merge directions 180 deg apart, as explained above

# The ellipse centre is in a different section of result.yaml than CALIB.
with open(trial_dir / "result.yaml", "r") as f:
    ELLIPSE_CENTRE = yaml.safe_load(f)["spatial_fine_calibration"]["real_ellipse"]["center"]

real_all = np.asarray(real_events)          # first cell that reads the event data
N_SEGMENTS = len(splits) - 1
N_WINDOWS  = PERIOD_US // WINDOW_US

direction = np.arctan2(real_all["y"].astype(np.float64) - ELLIPSE_CENTRE[1],
                       real_all["x"].astype(np.float64) - ELLIPSE_CENTRE[0])

# Which segment and which window inside it each event belongs to.
segment_of = np.searchsorted(splits, real_all["t"], side="right") - 1
window_of  = (real_all["t"] - splits[np.clip(segment_of, 0, N_SEGMENTS - 1)]) // WINDOW_US
in_range   = (segment_of >= 0) & (segment_of < N_SEGMENTS) & (window_of < N_WINDOWS)


def average_arrow(mask, double=True):
    """Average arrow for every (segment, window).

    Returns two arrays shaped N_SEGMENTS x N_WINDOWS: how far right the averaged
    arrow points, and how far up. Together they hold both the direction and how
    tightly the events agree on it (the arrow's length).
    """
    use = in_range & mask
    angle = 2.0 * direction[use] if double else direction[use]
    slot = segment_of[use] * N_WINDOWS + window_of[use]
    count = np.bincount(slot, minlength=N_SEGMENTS * N_WINDOWS).astype(float)
    count[count == 0] = 1.0
    right = np.bincount(slot, weights=np.cos(angle), minlength=N_SEGMENTS * N_WINDOWS) / count
    up    = np.bincount(slot, weights=np.sin(angle), minlength=N_SEGMENTS * N_WINDOWS) / count
    return right.reshape(N_SEGMENTS, N_WINDOWS), up.reshape(N_SEGMENTS, N_WINDOWS)


arrows = {name: average_arrow(mask, DOUBLE) for name, mask in
          (("ON", real_all["p"] > 0), ("OFF", real_all["p"] <= 0))}

print(f"window {WINDOW_US} us -> {N_WINDOWS} windows per segment, {N_SEGMENTS} segments")
print(f"doubling: {DOUBLE}")
for name, (right, up) in arrows.items():
    length = np.hypot(right, up)
    print(f"  {name:>3}: average arrow length {length.mean():.3f}   "
          f"(1 = every event in one direction, 0 = spread all round)")

window 2500 us -> 196 windows per segment, 30 segments
doubling: True
   ON: average arrow length 0.828   (1 = every event in one direction, 0 = spread all round)
  OFF: average arrow length 0.796   (1 = every event in one direction, 0 = spread all round)


In [27]:
# --- Plot 1: direction over time, with the split points ------------------------
SEGMENTS_SHOWN = 6

right, up = arrows[POLARITY]
# Halving undoes the doubling and turns the arrow back into a direction. It puts a
# jump back in at 180 deg, which is fine for looking at but is avoided in plot 2.
angle_deg = np.degrees(np.arctan2(up, right) / (2.0 if DOUBLE else 1.0))

shown = min(SEGMENTS_SHOWN, N_SEGMENTS)
time_ms = (splits[:shown, None] + (np.arange(N_WINDOWS)[None, :] + 0.5) * WINDOW_US) / 1000

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_ms.ravel(), y=angle_deg[:shown].ravel(), mode="lines",
    line=dict(color=MUTED, width=1.2), name=f"{POLARITY} direction",
))
for cut in splits[:shown + 1]:
    fig.add_vline(x=cut / 1000, line=dict(color=ORANGE, width=2))

fig.update_layout(
    title=f"Direction the events lie in, with split points &mdash; {TRIAL}, {POLARITY}, "
          f"first {shown} segments",
    xaxis_title="time (ms)", yaxis_title="direction (degrees)",
    template="plotly_white", height=400, showlegend=False,
    margin=dict(l=60, r=30, t=60, b=50),
)
fig.show()

In [28]:
# --- Plot 2: difference from a reference segment, across the segment ------------
REFERENCE_SEGMENT = 0

right, up = arrows[POLARITY]
position_ms = (np.arange(N_WINDOWS) + 0.5) * WINDOW_US / 1000


def difference_deg(a, b):
    """Signed angle from arrow a to arrow b, in degrees of wheel angle.

    Worked out from the arrows themselves rather than by subtracting two direction
    numbers, so there is no value where the answer jumps.
    """
    (ar, au), (br, bu) = a, b
    along  = ar * br + au * bu       # how much they point the same way
    across = ar * bu - au * br       # how much one is turned from the other
    return np.degrees(np.arctan2(across, along)) / (2.0 if DOUBLE else 1.0)


groups = {"odd number of segments apart": [], "even number of segments apart": []}
for k in range(N_SEGMENTS):
    if k == REFERENCE_SEGMENT:
        continue
    key = ("odd" if (k - REFERENCE_SEGMENT) % 2 else "even") + " number of segments apart"
    groups[key].append(difference_deg((right[REFERENCE_SEGMENT], up[REFERENCE_SEGMENT]),
                                      (right[k], up[k])))

fig = go.Figure()
for (label, curves), colour in zip(groups.items(), (ORANGE, BLUE)):
    xs, ys = [], []
    for curve in curves:
        xs.extend([*position_ms, None])
        ys.extend([*curve, None])
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", opacity=0.5,
                             name=f"{label} ({len(curves)})", line=dict(color=colour, width=1)))
fig.add_hline(y=0, line=dict(color=INK, width=1, dash="dot"))

for label, curves in groups.items():
    print(f"{label:<32}: mean size of difference "
          f"{np.mean([np.abs(c).mean() for c in curves]):6.1f} deg")

fig.update_layout(
    title=f"Difference from segment {REFERENCE_SEGMENT} &mdash; {TRIAL}, {POLARITY}",
    xaxis_title="position within segment (ms)",
    yaxis_title="difference in wheel angle (degrees)",
    template="plotly_white", height=440,
    margin=dict(l=60, r=30, t=60, b=50),
)
fig.show()

odd number of segments apart    : mean size of difference   84.0 deg
even number of segments apart   : mean size of difference    2.8 deg


### Start and end angle of every segment

If the splits are right, every segment should begin with the wheel in the same place and
end with it in the same place. So take the first window of each segment and the last, and
look at the direction in each.

Two numbers come out of this. How much the start angles disagree with each other says how
well the splits land. Whether they drift steadily from the first segment to the last says
whether the error builds up over the recording or just wanders.

Comparing angles is done through the arrows again rather than by subtracting the plotted
numbers, so nothing jumps.

In [29]:
# --- Start and end angle of every segment --------------------------------------
right, up = arrows[POLARITY]
halve = 2.0 if DOUBLE else 1.0

start_angle = np.degrees(np.arctan2(up[:, 0],  right[:, 0])) / halve
end_angle   = np.degrees(np.arctan2(up[:, -1], right[:, -1])) / halve
segment_index = np.arange(N_SEGMENTS)


def deviation_from_average(r, u):
    """How far each segment sits from the average of all of them, in wheel degrees."""
    mr, mu = r.mean(), u.mean()
    along, across = r * mr + u * mu, r * mu - u * mr
    return np.degrees(np.arctan2(across, along)) / halve


start_dev = deviation_from_average(right[:, 0],  up[:, 0])
end_dev   = deviation_from_average(right[:, -1], up[:, -1])
same_state = segment_index % 2 == 0          # segments an even number apart match

print(f"window {WINDOW_US} us, {POLARITY} events\n")
print(f"{'':<14}{'all segments':>15}{'even-index only':>18}")
for label, dev in (("start angle", start_dev), ("end angle", end_dev)):
    print(f"{label:<14}{dev.std():>13.1f} deg{dev[same_state].std():>15.1f} deg")

# Does the start angle walk steadily across the recording, or just wander?
slope = np.polyfit(segment_index[same_state], start_dev[same_state], 1)[0]
print(f"\ndrift of the start angle across the record: "
      f"{slope * (N_SEGMENTS - 1):+.1f} deg over {N_SEGMENTS} segments "
      f"({slope:+.2f} deg per segment)")

fig = go.Figure()
fig.add_trace(go.Scatter(x=segment_index, y=start_angle, mode="lines+markers",
                         name="start of segment", line=dict(color=BLUE, width=2),
                         marker=dict(size=7)))
fig.add_trace(go.Scatter(x=segment_index, y=end_angle, mode="lines+markers",
                         name="end of segment", line=dict(color=ORANGE, width=2),
                         marker=dict(size=7)))
fig.update_layout(
    title=f"Direction at the start and end of each segment &mdash; {TRIAL}, {POLARITY}",
    xaxis_title="segment index", yaxis_title="direction (degrees)",
    template="plotly_white", height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(l=60, r=30, t=70, b=50),
)
fig.show()

window 2500 us, ON events

                 all segments   even-index only
start angle            42.1 deg            1.2 deg
end angle              42.1 deg            4.1 deg

drift of the start angle across the record: -3.2 deg over 30 segments (-0.11 deg per segment)


# Step 5 Periodic Comparison

In [72]:
APERTURE_PERIODS_US = PERIOD_US
ROTATION_PERIODS_US = APERTURE_PERIODS_US * 4

# DATA_T_START = np.max([REAL_T_START, V2E_T_START])
# DATA_T_END = np.min([REAL_T_END, V2E_T_END])

splits = SPLITTERS[SPLITTER_NAME](real_events, REAL_T_START, REAL_T_END, period_us=ROTATION_PERIODS_US)
splits_point_count = len(splits)
segment_count = splits_point_count - 1

print(f"Generating splits with rotaiton period (2 apertures): {splits}")
print(f"This will generate {splits_point_count} split points and {segment_count} segments")

SEGMENT_BUFFER_DIR = PROJECT_ROOT / "output" / "segment_buffer" / TRIAL
SEGMENT_BUFFER_DIR.mkdir(parents=True, exist_ok=True)

Generating splits with rotaiton period (2 apertures): [    6159  1971527  3936895  5902263  7867631  9832999 11798367 13763735]
This will generate 8 split points and 7 segments


In [73]:
# Generate segments
import h5py

def generate_segment_with_hilo(input_data:h5py.Dataset, timestamps, split_start_t:int, split_end_t:int) -> np.ndarray:
    '''
    Generate a segment based on a give starting timestep and an ending timestamp
    '''
    lo, hi = np.searchsorted(timestamps, [split_start_t, split_end_t])
    return input_data[lo:hi]

def save_segment(segment:np.ndarray, data_source_type:str, i_seg:int) -> Path:
    '''
    Save one segment into SEGMENT_BUFFER_DIR, named by source, index and time range
    '''
    file_name = make_segment_buffer_name(TRIAL, data_source_type, i_seg)
    file_path = SEGMENT_BUFFER_DIR / file_name
    np.save(file_path, segment)
    return file_path

def make_segment_buffer_name(TRIAL, data_source_type, i_seg)->str:
    return f"{TRIAL}_{data_source_type}_seg{i_seg:03d}.npy"


def load_segment(data_source_type:str, i_seg:int) -> np.ndarray:
    '''
    Load one saved segment back from SEGMENT_BUFFER_DIR
    '''
    return np.load(SEGMENT_BUFFER_DIR / make_segment_buffer_name(TRIAL, data_source_type, i_seg))

def generate_and_save_segments(real_events, v2e_events, * , segment_count, splits):
    real_timestamps = real_events["t"]
    v2e_timestamps = v2e_events["t"]

    for i_seg in range(0, segment_count, 1):
        segment_start_t= splits[i_seg]
        segment_end_t = splits[i_seg+1]
        real_segment = generate_segment_with_hilo(real_events, real_timestamps, segment_start_t, segment_end_t)
        v2e_segment = generate_segment_with_hilo(v2e_events, v2e_timestamps, segment_start_t, segment_end_t)

        save_segment(real_segment, "real", i_seg)
        save_segment(v2e_segment, "v2e", i_seg)

        print(f"Generated and saved real/v2e data for segment {i_seg}, with start/end in t: {segment_start_t, segment_end_t}")

    

In [74]:
# Only run when we need to generate data.
generate_and_save_segments(real_events, v2e_events, segment_count=segment_count, splits=splits)

Generated and saved real/v2e data for segment 0, with start/end in t: (np.int64(6159), np.int64(1971527))
Generated and saved real/v2e data for segment 1, with start/end in t: (np.int64(1971527), np.int64(3936895))
Generated and saved real/v2e data for segment 2, with start/end in t: (np.int64(3936895), np.int64(5902263))
Generated and saved real/v2e data for segment 3, with start/end in t: (np.int64(5902263), np.int64(7867631))
Generated and saved real/v2e data for segment 4, with start/end in t: (np.int64(7867631), np.int64(9832999))
Generated and saved real/v2e data for segment 5, with start/end in t: (np.int64(9832999), np.int64(11798367))
Generated and saved real/v2e data for segment 6, with start/end in t: (np.int64(11798367), np.int64(13763735))


In [75]:
from event_analysis_toolbox.metrics import get_metric
from event_analysis_toolbox.feature_preprocessing import window_features

COMPARE_WINDOW_US = 1000

with open(PROJECT_ROOT / "config.yaml") as f:
    CFG = yaml.safe_load(f)
FEATURE_SCALES = CFG["feature_scales"]

METRIC_SELECTION = "mmd"


def make_metric(metric_name:str, kernel_max_distance:float=None):
    '''
    Build a metric and its settings from config.yaml.
    kernel_max_distance replaces the MMD kernel width when given.
    '''
    settings = {metric_name: dict(CFG[metric_name])}
    if kernel_max_distance is not None:
        settings[metric_name]["kernels"] = [{
            "rbf_kernel_max_distance": kernel_max_distance,
            "rbf_kernel_target_similarity": 0.5,
        }]
    metric = get_metric(metric_name)
    metric_settings = metric.build_kwargs(settings)
    if metric.supports_inner_progress:
        metric_settings["progress"] = False
    return metric, metric_settings


def segment_to_windows(segment:np.ndarray, segment_start_t:int, segment_end_t:int,
                       window_us:int=COMPARE_WINDOW_US) -> list:
    '''
    Cut one segment into fixed length windows, each ready to hand to a metric.
    Time is measured from each window's own start and every axis is scaled by config.yaml.
    '''
    window_starts = np.arange(segment_start_t, segment_end_t, window_us)
    lo = np.searchsorted(segment["t"], window_starts)
    hi = np.searchsorted(segment["t"], window_starts + window_us)
    return [window_features(segment[a:b], feature_scales=FEATURE_SCALES, time_origin=start)[0]
            for a, b, start in zip(lo, hi, window_starts)]


def compare_to_baseline(baseline_windows:list, other_windows:list,
                        metric, metric_settings:dict) -> np.ndarray:
    '''
    Distance between matching windows of two segments: window 5 against window 5.
    Windows holding fewer than 2 events give not-a-number instead of a distance.
    '''
    distances = np.full(len(baseline_windows), np.nan)
    for i_window, (baseline, other) in enumerate(zip(baseline_windows, other_windows)):
        if len(baseline) >= 2 and len(other) >= 2:
            distances[i_window] = metric.compute(baseline, other, **metric_settings).value
    return distances

In [79]:
# --- Run a comparison and save it ----------------------------------------------
import pandas as pd

RESULT_DIR = PROJECT_ROOT / "output" / "period_comparison" / TRIAL
DISTANCE_DIR = RESULT_DIR / "distances"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
DISTANCE_DIR.mkdir(parents=True, exist_ok=True)

# --- choose what to run --------------------------------------------------------
METRIC_NAME = "mmd"              # "mmd" or "sliced_wasserstein"
KERNEL_MAX_DISTANCE = 15         # MMD kernel width; None for sliced_wasserstein
BASELINE_SEGMENT = 0
COMPARE_SOURCES = ("real", "v2e")


def make_result_name(metric_name:str, kernel_max_distance:float=None) -> str:
    '''
    Result file name, carrying the metric, its kernel width and the window length
    '''
    kernel_tag = "" if kernel_max_distance is None else f"_rbf{int(kernel_max_distance):02d}"
    return f"{TRIAL}_{metric_name}{kernel_tag}_w{COMPARE_WINDOW_US}us.csv"


def make_distance_name(metric_name:str, kernel_max_distance:float, baseline_segment:int,
                       data_source_type:str, i_seg:int) -> str:
    '''
    File name for the distances of one comparison: baseline against one segment
    '''
    kernel_tag = "" if kernel_max_distance is None else f"_rbf{int(kernel_max_distance):02d}"
    return (f"{TRIAL}_{metric_name}{kernel_tag}_w{COMPARE_WINDOW_US}us"
            f"_base{baseline_segment:03d}_{data_source_type}_seg{i_seg:03d}.csv")


def run_comparison(metric_name:str, kernel_max_distance:float=None, *,
                   baseline_segment:int=0,
                   compare_sources=("real", "v2e")) -> pd.DataFrame:
    '''
    Compare every segment against one real baseline segment, window by window.
    Returns one row per window and writes the same table to RESULT_DIR as a csv.
    '''
    metric, metric_settings = make_metric(metric_name, kernel_max_distance)
    baseline_windows = segment_to_windows(load_segment("real", baseline_segment),
                                          splits[baseline_segment],
                                          splits[baseline_segment + 1])
    blocks = []
    for i_seg in range(segment_count):
        for data_source_type in compare_sources:
            if data_source_type == "real" and i_seg == baseline_segment:
                continue                      # a segment against itself is zero
            windows = segment_to_windows(load_segment(data_source_type, i_seg),
                                         splits[i_seg], splits[i_seg + 1])
            distances = compare_to_baseline(baseline_windows, windows, metric, metric_settings)
            window_index = np.arange(len(distances))
            block = pd.DataFrame({
                "compare_source": data_source_type,
                "compare_segment": i_seg,
                "window_index": window_index,
                "window_start_us": splits[i_seg] + window_index * COMPARE_WINDOW_US,
                "distance": distances,
            })
            block.to_csv(DISTANCE_DIR / make_distance_name(
                metric_name, kernel_max_distance, baseline_segment,
                data_source_type, i_seg), index=False)
            blocks.append(block)
            print(f"Segment {i_seg}, {data_source_type}, distances calculated. Sum: {np.sum(distances)}, Mean: {np.mean(distances)}")
        print(f"segment {i_seg:2d} done")

    table = pd.concat(blocks, ignore_index=True)
    table.insert(0, "trial", TRIAL)
    table.insert(1, "metric", metric_name)
    table.insert(2, "kernel_max_distance", kernel_max_distance)
    table.insert(3, "baseline_segment", baseline_segment)

    file_path = RESULT_DIR / make_result_name(metric_name, kernel_max_distance)
    table.to_csv(file_path, index=False)
    print(f"\nwrote {file_path.name}  ({len(table):,} rows)")
    print(f"and {len(blocks)} per-comparison files in {DISTANCE_DIR.name}/")
    return table

table = run_comparison(METRIC_NAME, KERNEL_MAX_DISTANCE,
                       baseline_segment=BASELINE_SEGMENT,
                       compare_sources=COMPARE_SOURCES)

Segment 0, v2e, distances calculated. Sum: 219.89425001187166, Mean: 0.11184855036209139
segment  0 done
Segment 1, real, distances calculated. Sum: 83.15958117666091, Mean: 0.0422988714021673
Segment 1, v2e, distances calculated. Sum: 230.40478472378237, Mean: 0.11719470230100833
segment  1 done
Segment 2, real, distances calculated. Sum: 127.42850760743588, Mean: 0.06481612797936719
Segment 2, v2e, distances calculated. Sum: 260.3699383602304, Mean: 0.1324363877722433
segment  2 done
Segment 3, real, distances calculated. Sum: 84.37127041873957, Mean: 0.042915193498850236
Segment 3, v2e, distances calculated. Sum: 227.29525804306763, Mean: 0.11561305088660612
segment  3 done
Segment 4, real, distances calculated. Sum: 87.91963008457249, Mean: 0.04472005599418743
Segment 4, v2e, distances calculated. Sum: 214.94808746818865, Mean: 0.10933269962776634
segment  4 done
Segment 5, real, distances calculated. Sum: 111.68944119478734, Mean: 0.056810499081784
Segment 5, v2e, distances calcul

In [80]:
# --- Read a saved result and plot it -------------------------------------------
PLOT_METRIC_NAME = "mmd"
PLOT_KERNEL_MAX_DISTANCE = 15

result_path = RESULT_DIR / make_result_name(PLOT_METRIC_NAME, PLOT_KERNEL_MAX_DISTANCE)
table = pd.read_csv(result_path)

summary = (table.groupby(["compare_source", "compare_segment"])["distance"]
                .agg(["sum", "mean"]).reset_index())
print(f"{result_path.name}\n")
print(summary.to_string(index=False))

print("\nmean over all segments:")
for data_source_type, group in summary.groupby("compare_source"):
    print(f"  {data_source_type:>4} vs real baseline: {group['mean'].mean():.5f}")

fig = go.Figure()
for data_source_type, colour in (("real", BLUE), ("v2e", ORANGE)):
    group = summary[summary.compare_source == data_source_type]
    if group.empty:
        continue
    fig.add_trace(go.Scatter(x=group.compare_segment, y=group["mean"],
                             mode="lines+markers", name=f"{data_source_type} vs real baseline",
                             line=dict(color=colour, width=2), marker=dict(size=9)))
fig.update_layout(
    title=f"Mean distance to real segment {table.baseline_segment.iloc[0]}"
          f" &mdash; {TRIAL}, {PLOT_METRIC_NAME}"
          + ("" if PLOT_KERNEL_MAX_DISTANCE is None else f" rbf{PLOT_KERNEL_MAX_DISTANCE}"),
    xaxis_title="segment index", yaxis_title="mean distance",
    template="plotly_white", height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    margin=dict(l=60, r=30, t=70, b=50))
fig.show()

f1_mmd_rbf15_w1000us.csv

compare_source  compare_segment        sum     mean
          real                1  83.159581 0.042299
          real                2 127.428508 0.064816
          real                3  84.371270 0.042915
          real                4  87.919630 0.044720
          real                5 111.689441 0.056810
          real                6 175.085760 0.089057
           v2e                0 219.894250 0.111849
           v2e                1 230.404785 0.117195
           v2e                2 260.369938 0.132436
           v2e                3 227.295258 0.115613
           v2e                4 214.948087 0.109333
           v2e                5 212.712418 0.108196
           v2e                6 231.453774 0.117728

mean over all segments:
  real vs real baseline: 0.05677
   v2e vs real baseline: 0.11605
